# 00 · Verificações de qualidade

Fase 2 do [`docs/workflow.md`](../docs/workflow.md). Resultados e decisões nas secções 1 a 6 de [`docs/qualidade_dados.md`](../docs/qualidade_dados.md).

**Pergunta geral:** os dados são usáveis? Há linhas vazias, repetidas ou com formatos estranhos?

**Porque usa todos os dados, incluindo o teste:** estas verificações só contam problemas técnicos (campos vazios, duplicados, formatos). Corrigi-los não depende de ver os resultados, por isso não é data snooping. A única exceção são os exemplos de texto mostrados na secção 6, que vêm só de antes de novembro de 2025.

**Como correr:** no VS Code, kernel do `.venv`, "Run All". Ou no terminal: `.venv/bin/jupyter nbconvert --to notebook --execute --inplace notebooks/00_qualidade.ipynb`

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import cpv_data as cd

def pct(n, total):
    return f"{n:,} ({n / total:.1%})".replace(",", " ")

def preenchido(s):
    return s.notna() & (s.str.strip() != "")

df = cd.load_contracts(["objectoContrato", "descContrato", "CPV"])
total = len(df)
print("Contratos nos ficheiros de 2025 e 2026:", f"{total:,}".replace(",", " "))

Contratos nos ficheiros de 2025 e 2026: 433 838


## 1 e 4. Contratos repetidos (mesmo `idcontrato`)

**Como ler:** se as cópias tiverem o mesmo texto e o mesmo CPV, basta ficar com uma. Se fossem diferentes, teríamos de decidir qual é a certa.

In [2]:
print("Linhas com idcontrato repetido:", pct(df["idcontrato"].duplicated().sum(), total))
dup = df[df["idcontrato"].duplicated(keep=False)]
iguais = dup.groupby("idcontrato")[["objectoContrato", "CPV"]].nunique().max(axis=1) == 1
print("Cópias com o mesmo texto e CPV:", iguais.sum(), "de", len(iguais))

Linhas com idcontrato repetido: 1 199 (0.3%)
Cópias com o mesmo texto e CPV: 1187 de 1187


## 2. Datas de publicação

**Como ler:** todas as datas têm de ser válidas, porque é por elas que separamos treino, validação e teste. E mostra que o ficheiro de um ano não é o mesmo que o ano de publicação.

In [3]:
print("Sem data válida:", pct(df["data"].isna().sum(), total))
df["data"].dt.year.value_counts().sort_index().rename("contratos").to_frame()

Sem data válida: 0 (0.0%)


,contratos
data,
2024,6
2025,225113
2026,208719


## 1. Campos preenchidos

**Como ler:** um contrato sem texto ou sem CPV não serve para treinar. A última linha mostra se a `descContrato` traz alguma coisa além do `objectoContrato`.

In [4]:
obj, desc, cpv = preenchido(df["objectoContrato"]), preenchido(df["descContrato"]), preenchido(df["CPV"])
print("objectoContrato:", pct(obj.sum(), total))
print("descContrato:   ", pct(desc.sum(), total))
print("CPV:            ", pct(cpv.sum(), total))
print("objecto + CPV:  ", pct((obj & cpv).sum(), total))
iguais = cd.normalize_text(df["objectoContrato"]) == cd.normalize_text(df["descContrato"])
print("descContrato igual ao objectoContrato:", pct(iguais.sum(), total))

objectoContrato: 433 838 (100.0%)
descContrato:    433 838 (100.0%)
CPV:             433 835 (100.0%)
objecto + CPV:   433 835 (100.0%)


descContrato igual ao objectoContrato: 313 691 (72.3%)


## 3. Códigos por célula de CPV

**Como ler:** se quase todos os contratos tiverem um só código, a regra "usar o primeiro" chega. Se muitos tivessem vários códigos de divisões diferentes, teríamos um problema de várias etiquetas por contrato.

In [5]:
codigos = cd.all_codes(df.loc[cpv, "CPV"])
n_codigos = codigos.map(len)
n_cpv = int(cpv.sum())
for k, n in n_codigos.clip(upper=5).value_counts().sort_index().items():
    print(f"{k if k < 5 else '5 ou mais'} código(s):", pct(n, n_cpv))
varios = codigos[n_codigos > 1]
mesma_div = varios.map(lambda cs: len({c[:2] for c in cs}) == 1)
print("Vários, mesma divisão:      ", pct(mesma_div.sum(), n_cpv))
print("Vários, divisões diferentes:", pct((~mesma_div).sum(), n_cpv))
print("Sem código reconhecido:     ", (n_codigos == 0).sum())

1 código(s): 428 593 (98.8%)
2 código(s): 4 282 (1.0%)
3 código(s): 514 (0.1%)
4 código(s): 196 (0.0%)
5 ou mais código(s): 250 (0.1%)
Vários, mesma divisão:       1 317 (0.3%)
Vários, divisões diferentes: 3 925 (0.9%)
Sem código reconhecido:      0


## 5 e 6. Textos duplicados

**Como ler:** muitos textos repetidos querem dizer duas coisas: risco de fuga de dados (o mesmo texto no treino e no teste) e textos que podem não dizer o que se compra. Os textos repetidos com códigos diferentes mostram contradições nas etiquetas.

In [6]:
base = df.loc[obj & cpv].copy()
base["texto"] = cd.normalize_text(base["objectoContrato"])
base["primeiro"] = cd.first_code(base["CPV"])
n_base = len(base)
print("Contratos com texto repetido:", pct(base["texto"].duplicated().sum(), n_base))
print("Textos distintos:            ", pct(base["texto"].nunique(), n_base))
por_texto = base.groupby("texto")["primeiro"].nunique()
conflito = por_texto[por_texto > 1]
print("Textos repetidos com códigos diferentes:", f"{len(conflito):,}".replace(",", " "),
      f"(cobrem {base['texto'].isin(conflito.index).sum():,} contratos)".replace(",", " "))

Contratos com texto repetido: 139 064 (32.1%)


Textos distintos:             294 771 (67.9%)


Textos repetidos com códigos diferentes: 5 220 (cobrem 39 972 contratos)


In [7]:
print("Textos mais repetidos, só antes de novembro de 2025:")
top = base.loc[base["data"] < cd.TEST_START, "texto"].value_counts().head(5)
top.index = top.index.str[:80]  # cortar só para mostrar, depois de contar
top.to_frame("contratos")

Textos mais repetidos, só antes de novembro de 2025:


,contratos
texto,
medicamentos,1870
aquisição de serviço de viagens e alojamento – plataforma siav,722
convite à apresentação de propostas ao abrigo do(s) acordo(s)-quadro 2023/304 -,429
1396/2025-convite à apresentação de propostas ao abrigo do(s) acordo(s)-quadro 2,421
1411/2025 - convite à apresentação de propostas ao abrigo dos acordos-quadro: 20,402
